In [1]:
%load_ext autoreload
%autoreload 2

# Semantic Search with Toponymy

## Why Explore Topics?

When working with large document collections, traditional search methods—keyword search, 
semantic similarity—are powerful but limited. They work well when you know what you're 
looking for, but they don't help you *discover* what's in your corpus.

Topic modeling complements traditional search by organizing documents into thematic 
clusters. A topic hierarchy reveals:
- What themes exist in your collection
- How topics relate to one another
- Which documents belong to broader or narrower topics

## Combining Search with Topic Exploration

`TopicTreeSearch` bridges the gap between traditional search and topic discovery. It lets 
you:

1. **Search** using keywords or semantic similarity (like traditional search)
2. **See results in context** of the topic hierarchy (topic-aware exploration)
3. **Drill down** into topics to understand what documents they contain
4. **Navigate** between search results and the broader topic structure

This makes it easier to:
- Find documents you're looking for while understanding the broader topical context
- Discover related documents you might have missed with traditional search alone
- Understand how your search results fit into the document's thematic landscape

## This Notebook

In this notebook, we'll fit a Toponymy topic model on the 20 Newsgroups dataset and 
interactively explore it using both keyword and semantic search. You'll see how the topic 
hierarchy provides a useful lens for understanding your corpus, whether you're searching 
for specific documents or discovering new themes.

## Important Note: This is a Demonstration

The `TopicTreeSearch.explorer()` widget in this notebook is a **proof-of-concept 
demonstration**, not a production-ready tool. It's designed to quickly show what 
topic-aware search could enable in Toponymy, not to serve as a definitive explorer.

**Limitations of this example:**
- Uses brute-force cosine similarity (no approximate nearest neighbor algorithms)
- Not optimized for performance on large datasets
- Generates interactive ipywidgets for notebooks only (no HTML export)
- Intended for exploration and prototyping, not as a finished product

**This is educational code.** It demonstrates the *concept* and helps you understand 
what's possible with Toponymy. Future versions could be enhanced with:
- More efficient search algorithms
- Static HTML export
- Additional visualization options
- Production-grade performance optimization

More scalable versions should leverage approximate nearest neighbor algorithms 
(such as FAISS, Annoy, or pynndescent) for efficient search on large corpora.

Use this notebook to explore how topic-aware search works, then adapt the ideas 
to your own use cases!

## Setup: Install Dependencies

This notebook uses two components:

**Embedding Model** (for semantic search): We use `sentence-transformers`, a popular 
open-source library that runs locally on CPU and doesn't require API keys. You can substitute 
with commercial alternatives like `OpenAIEmbedder` with minimal code changes.

**Topic Naming** (for readable topic labels): Topic names are generated using Toponymy's 
LLM-based naming, which requires an LLM API (this notebook uses OpenAI). Topic names are 
essential for making the topic tree readable and explorable.

### API Key Configuration

Before running this notebook, ensure your OpenAI API key is available:

- **Recommended**: Set `OPENAI_API_KEY` in your environment (`.bashrc`, `.env` file, etc...)
- **Alternative**: Pass the key directly to the `OpenAINamer` class
- **Important**: Never hardcode API keys in notebooks—they can leak to version control

On shared Jupyter Hubs, note that `.bashrc` variables won't be available; use a `.env` file 
or pass the key directly instead.

This setup keeps semantic search fast (CPU-friendly local embeddings) while using an 
LLM API for readable topic naming.

First we ensure our API Keys are loaded.

In [2]:
import os
from dotenv import load_dotenv

# Load from .env file if it exists
load_dotenv()

# Verify API key is available
if not os.getenv("OPENAI_API_KEY"):
    raise ValueError(
        "OPENAI_API_KEY not found. Set it in your environment or .env file. "
        "See the Setup section above for details."
    )

Load the necessary libraries

In [3]:
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from toponymy.llm_wrappers import OpenAINamer
from toponymy import Toponymy, ToponymyClusterer
from toponymy.plotting import TopicTreeSearch
from toponymy.tools.notebook_data_load import load_newsgroups

Instantiate our embedding and naming models and ensure that the naming model has connected properly using your API Key.

In [4]:
embedder = SentenceTransformer("all-mpnet-base-v2")
namer = OpenAINamer()
namer.test_llm_connectivity()

'{"status": "ok"}'

## Step 1: Load the 20 Newsgroups Dataset

We'll use the 20 Newsgroups dataset from scikit-learn, which contains ~18,000 
documents organized into 20 categories. For this interactive demonstration, we'll 
sample it down to 150 documents for faster exploration. To explore the full corpus, 
set use_small=False below.

In [5]:
import numpy as np
import pandas as pd
from toponymy.tools.notebook_data_load import load_newsgroups

df = load_newsgroups(use_small=False)
documents = df["post"].values
newsgroups = df["newsgroup"].values
document_map = np.stack(df["map"].values)
document_vectors = np.stack(df["embedding"].values)


## Step 2: Build and Fit Toponymy

To explore documents through a topic lens (as discussed above), we need to first 
build a topic hierarchy. Toponymy organizes documents into a hierarchical topic 
tree, which is what we'll use to enhance our interactive search and exploration.

In this step we fit a Toponymy model on the newsgroups documents and generate 
topic names. This gives us the topic hierarchy we need for topic-aware exploration.

It can be useful to cluster first to ensure we are naming topics at a useful resolution.
A user should alter the paramters here depending on the size of the data set being explored.

In [6]:
clusterer = ToponymyClusterer(verbose=False, min_clusters=6)
clusterer.fit(clusterable_vectors=document_map, embedding_vectors=document_vectors, verbose=True);

Layer 0 found 428 clusters
Layer 1 found 134 clusters
Layer 2 found 41 clusters
Layer 3 found 14 clusters


In [7]:
from toponymy import Toponymy, ToponymyClusterer

toponymy = Toponymy(
    llm_wrapper=namer,
    text_embedding_model=embedder,  # Pass our 768-dim embedder
    clusterer=clusterer,
    object_description="newsgroup posts",
    corpus_description="20-newsgroups dataset",
)

toponymy.fit(
    documents,
    embedding_vectors=document_vectors,
    clusterable_vectors=document_map,  # Use UMAP coordinates for clustering
)

Selecting central exemplars:   0%|          | 0/428 [00:00<?, ?cluster/s]

Batches:   0%|          | 0/1563 [00:00<?, ?it/s]

Building topic names by layer:   0%|          | 0/4 [00:00<?, ?layer/s]

Generating informative keyphrases:   0%|          | 0/428 [00:00<?, ?cluster/s]

Generating prompts for layer 0:   0%|          | 0/428 [00:00<?, ?topic/s]

...

Generating informative keyphrases:   0%|          | 0/14 [00:00<?, ?cluster/s]

Selecting central subtopics:   0%|          | 0/14 [00:00<?, ?cluster/s]

Generating prompts for layer 3:   0%|          | 0/14 [00:00<?, ?topic/s]

Generating topic names for layer 3:   0%|          | 0/14 [00:00<?, ?topic/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [18]:
unlabelled= np.sum(toponymy.topic_name_vectors_[-1]=='Unlabelled')
total = len(toponymy.topic_name_vectors_[-1])

print(unlabelled/total)

0.2453494771601541


## Step 3: Interactive Topic Tree Explorer

This is where topic-aware exploration comes to life. The `TopicTreeSearch.explorer()` 
widget combines the best of both worlds:

- **Traditional search**: Keyword and semantic queries (like any search engine)
- **Topic context**: See how results fit into the hierarchical topic structure
- **Discovery**: Explore which topics your results belong to, find related documents 
  at broader or narrower topic levels

Run the cell below to launch the interactive explorer. Try:
1. **Keyword search**: Search for specific topics ("space", "religion", "baseball")
2. **Semantic search**: Use natural language queries ("Is there intelligent life?", 
   "How do I cook a steak?")
3. **Topic search**: Search for topics by name or semantic similarity
4. **Expand topics**: Click on topics to see which documents belong to them
5. **Export results**: Save matched documents to a JSON file for further analysis

This widget demonstrates the concept of topic-aware exploration. In a production 
setting, you might build a more robust tool with better performance and additional 
features—but the core idea is the same: combine search with topic discovery.

### Creating the Explorer

Run the cell below to launch the interactive widget. 

*Note:* the embedder must be 
the same model used to create the document embeddings (in this case, "all-mpnet-base-v2"). 
Using a different embedder will produce incorrect semantic search results.

In [8]:
search = TopicTreeSearch(toponymy, documents)
widget = search.explorer(embedder=embedder)
widget


For programmatic control, TopicTreeSearch also exposes:
- .keyword(query), .semantic(query) for document search
- .topic_keyword(query), .topic_semantic(query) for topic search
- .hierarchy(), .html() for custom visualizations

See the TopicTreeSearch API documentation for examples.

## Key Takeaways

We've just seen how topic-aware exploration can enhance traditional search:

- **Better discovery**: Instead of just finding documents, you see them in their 
  topical context
- **Navigable results**: Explore related documents by navigating the topic hierarchy
- **Flexible search**: Combine keyword and semantic search with topic-level insights
- **Intuitive interface**: The explorer widget makes this accessible to non-technical users

This was a proof-of-concept demonstration of what's possible. In production, you might 
build on these ideas with better performance, different visualization options, or 
integration with your own domain-specific document features.